# 02 — Run Tests Inside a Databricks Notebook

This notebook shows the Databricks-native approach: run pytest **inside a notebook cell** on a cluster.

This is useful when:
- You want to test against the cluster's SparkSession (not local Spark)
- You need access to Unity Catalog tables or volumes
- You're running integration tests that touch real infrastructure

**To run this notebook:** upload the entire `unit-testing-pyspark/` folder to your Databricks workspace and attach to a cluster (or use Serverless).

## Method 1: pytest from a notebook cell

This is the approach Databricks recommends. Install pytest, point it at your test files, done.

In [1]:
# On a Databricks cluster, install pytest first
# Uncomment the next line when running on Databricks:
# %pip install pytest

In [2]:
import pytest
import sys

# Prevent pytest from writing .pyc / __pycache__ (cleaner on DBFS)
sys.dont_write_bytecode = True

# Run the full test suite
retcode = pytest.main([
    "tests/",
    "-v",
    "-p", "no:cacheprovider"
])

assert retcode == 0, "Tests failed! Check the output above."

============================= test session starts ==============================
platform darwin -- Python 3.12.10, pytest-9.0.2, pluggy-1.6.0 -- /Users/alexander.booth/miniconda3/envs/demo-env/bin/python
rootdir: /Users/alexander.booth/Documents/Repos/databricks_demos/unit-testing-pyspark
plugins: anyio-4.9.0, Faker-40.1.2
collecting ... collected 39 items

tests/test_pure_python.py::TestBattingAverage::test_normal_calculation PASSED [  2%]
tests/test_pure_python.py::TestBattingAverage::test_perfect_average PASSED [  5%]
tests/test_pure_python.py::TestBattingAverage::test_zero_at_bats_returns_zero PASSED [  7%]
tests/test_pure_python.py::TestBattingAverage::test_hitless PASSED       [ 10%]
tests/test_pure_python.py::TestBattingAverage::test_rounding PASSED      [ 12%]
tests/test_pure_python.py::TestSluggingPercentage::test_singles_only PASSED [ 15%]
tests/test_pure_python.py::TestSluggingPercentage::test_all_home_runs PASSED [ 17%]
tests/test_pure_python.py::TestSluggingPercentage::te

## Method 2: Inline tests in the notebook

For quick validation, you can write tests directly in notebook cells.
This is less structured but useful for ad-hoc checking.

In [3]:
from baseball_stats import batting_average, era, classify_hitter

# Inline assertions — these will raise AssertionError if they fail
assert batting_average(150, 500) == 0.300, "BA calculation broken"
assert batting_average(0, 0) == 0.0, "Zero division not handled"
assert era(30, 100.0) == 2.70, "ERA calculation broken"
assert classify_hitter(0.320) == "Elite", "Classification broken"

print("All inline assertions passed!")

All inline assertions passed!


## Method 3: Test with the cluster's SparkSession

When running on Databricks, the notebook already has a SparkSession (`spark`).
You can test transformations using it directly — useful for integration tests
that touch Unity Catalog.

When running locally with Databricks Connect, `DatabricksSession` gives you a
remote session connected to your cluster or serverless compute.

In [4]:
from databricks.connect import DatabricksSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from baseball_stats import add_batting_average, filter_qualified_batters, top_n_by_stat

# Use DatabricksSession with serverless — works both in notebooks and locally via Databricks Connect
spark = DatabricksSession.builder.serverless(True).getOrCreate()

# Create test data
schema = StructType([
    StructField("player",    StringType(),  True),
    StructField("team",      StringType(),  True),
    StructField("hits",      IntegerType(), True),
    StructField("at_bats",   IntegerType(), True),
    StructField("singles",   IntegerType(), True),
    StructField("doubles",   IntegerType(), True),
    StructField("triples",   IntegerType(), True),
    StructField("home_runs", IntegerType(), True),
])

data = [
    ("Mike Trout",      "Angels",  180, 550, 100, 30, 5,  45),
    ("Shohei Ohtani",   "Dodgers", 190, 500, 95,  35, 3,  57),
    ("Mookie Betts",    "Dodgers", 170, 520, 90,  40, 4,  36),
    ("Aaron Judge",     "Yankees", 160, 480, 75,  25, 2,  58),
    ("Ronald Acuna Jr", "Braves",  175, 540, 100, 35, 5,  35),
]

df = spark.createDataFrame(data, schema)
print("Raw data:")
df.show()

Raw data:
+---------------+-------+----+-------+-------+-------+-------+---------+
|         player|   team|hits|at_bats|singles|doubles|triples|home_runs|
+---------------+-------+----+-------+-------+-------+-------+---------+
|     Mike Trout| Angels| 180|    550|    100|     30|      5|       45|
|  Shohei Ohtani|Dodgers| 190|    500|     95|     35|      3|       57|
|   Mookie Betts|Dodgers| 170|    520|     90|     40|      4|       36|
|    Aaron Judge|Yankees| 160|    480|     75|     25|      2|       58|
|Ronald Acuna Jr| Braves| 175|    540|    100|     35|      5|       35|
+---------------+-------+----+-------+-------+-------+-------+---------+



In [5]:
# Run transformations and verify
result = add_batting_average(df)
print("With batting average added:")
result.show()

# Verify Trout's BA
trout_avg = result.filter(result.player == "Mike Trout").collect()[0].batting_avg
assert trout_avg == 0.327, f"Expected 0.327, got {trout_avg}"
print(f"Trout's BA: {trout_avg} ✓")

With batting average added:
+---------------+-------+----+-------+-------+-------+-------+---------+-----------+
|         player|   team|hits|at_bats|singles|doubles|triples|home_runs|batting_avg|
+---------------+-------+----+-------+-------+-------+-------+---------+-----------+
|     Mike Trout| Angels| 180|    550|    100|     30|      5|       45|      0.327|
|  Shohei Ohtani|Dodgers| 190|    500|     95|     35|      3|       57|       0.38|
|   Mookie Betts|Dodgers| 170|    520|     90|     40|      4|       36|      0.327|
|    Aaron Judge|Yankees| 160|    480|     75|     25|      2|       58|      0.333|
|Ronald Acuna Jr| Braves| 175|    540|    100|     35|      5|       35|      0.324|
+---------------+-------+----+-------+-------+-------+-------+---------+-----------+

Trout's BA: 0.327 ✓


In [6]:
# Filter and rank
qualified = filter_qualified_batters(result, min_at_bats=500)
print(f"Qualified batters (500+ AB): {qualified.count()} of {result.count()}")

top_hr = top_n_by_stat(df, "home_runs", n=3)
print("\nTop 3 HR leaders:")
top_hr.select("player", "team", "home_runs").show()

Qualified batters (500+ AB): 4 of 5

Top 3 HR leaders:
+-------------+-------+---------+
|       player|   team|home_runs|
+-------------+-------+---------+
|  Aaron Judge|Yankees|       58|
|Shohei Ohtani|Dodgers|       57|
|   Mike Trout| Angels|       45|
+-------------+-------+---------+



## Summary: Three Ways to Test

| Method | Where | When to use |
|---|---|---|
| `pytest tests/ -v` in terminal | Your laptop (via Databricks Connect) | Development loop, CI/CD |
| `pytest tests/ -v` in a cell | Databricks notebook | Structured test suite on a cluster |
| Inline `assert` statements | Databricks notebook | Quick ad-hoc validation |